In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

import sys
import os
import joblib
from pathlib import Path

root_path = os.path.abspath("..")
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data_preprocessing import clean_data
from src.feature_engineering import add_features
from src.modeling import train_model
from src.evaluation import evaluate_classification


1. Load Cleaned Dataset

In [2]:
DATA_DIR = Path.cwd().parent / "data"
file_path = DATA_DIR / "HR_capstone_dataset.csv"

In [3]:
df_raw = pd.read_csv(file_path)

In [4]:
df_raw.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


2. Split

In [5]:
X_raw = df_raw.drop("left", axis=1)
y_raw = df_raw["left"]

In [6]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_raw
)

3. Preprocessing & Feature Engineering

In [7]:
train_temp = pd.concat([X_train_raw, y_train], axis=1)
test_temp = pd.concat([X_test_raw, y_test], axis=1)

df_train_processed = add_features(clean_data(train_temp))
X_train = df_train_processed.drop("left", axis=1)
y_train = df_train_processed["left"].astype(int).values

df_test_processed = add_features(clean_data(test_temp))
X_test = df_test_processed.drop("left", axis=1)
y_test = df_test_processed["left"].astype(int).values

4. Train model

In [8]:
model = train_model(X_train, y_train)

5. Evaluate

In [9]:
evaluate_classification(model, X_test, y_test)

Model Performance:
Accuracy: 0.9805
Recall:   0.9215

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2232
           1       0.99      0.92      0.95       637

    accuracy                           0.98      2869
   macro avg       0.98      0.96      0.97      2869
weighted avg       0.98      0.98      0.98      2869



6. Save model

In [10]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/attrition_model.pkl")

['../models/attrition_model.pkl']

In [11]:
!pip freeze > ../requirements.txt
print("Model and Requirements saved successfully!")

Model and Requirements saved successfully!
